# Highest-Grossing Films — Data Extraction & Database Population

This notebook performs the following steps:
1. Fetches the Wikipedia page *List of highest-grossing films*.
2. Parses the all-time ranking table using **BeautifulSoup**.
3. Enriches each entry (director, country) by visiting individual film pages.
4. Cleans and normalises the raw text values.
5. Stores the result in a **SQLite** database (`films.db`).
6. Exports the database contents to **`films.json`** for the static web page.

## 1. Install & Import Dependencies

In [3]:
# Install required libraries (uncomment if not already installed)
%pip install requests beautifulsoup4 lxml

import requests
from bs4 import BeautifulSoup
import sqlite3
import json
import re
import time

print('All libraries imported successfully.')

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 8.6 MB/s  0:00:01 eta 0:00:01
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [beautifulsoup4]m [requests]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
All libraries imported successfully.


## 2. Fetch the Wikipedia Page

In [4]:
WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_highest-grossing_films'

# Identify the request as an educational bot to be a good citizen
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (educational assignment; data wrangling course)'
}

response = requests.get(WIKI_URL, headers=HEADERS, timeout=15)
response.raise_for_status()  # Raise an error for bad HTTP status codes

soup = BeautifulSoup(response.content, 'lxml')
print(f'Page fetched successfully. Status: {response.status_code}')
print(f'Page title: {soup.find("title").get_text()}')

Page fetched successfully. Status: 200
Page title: List of highest-grossing films - Wikipedia


## 3. Parse the All-Time Highest-Grossing Films Table

The Wikipedia table may contain extra columns (e.g. a "Peak" or reference column) and
cells with `rowspan` attributes. To handle both issues robustly we:
1. Read the header row and locate each column by keyword.
2. Reconstruct each data row's full column list by injecting placeholder cells for
   positions covered by a `rowspan` from the row above.

In [5]:
def parse_main_table(soup):
    """
    Extract rank, title, gross revenue, year, and the film's wiki link.

    Handles tables with extra columns and rowspan cells by:
    - detecting column positions from the header row, and
    - rebuilding each row with rowspan placeholders so indices stay consistent.
    """
    table = soup.find('table', {'class': 'wikitable'})
    if table is None:
        raise ValueError('Could not find the main wikitable on the page.')

    rows = table.find_all('tr')

    # ── Step 1: Detect column indices from the header row ─────────────────────
    header_cells = rows[0].find_all(['th', 'td'])
    headers = [c.get_text(strip=True).lower() for c in header_cells]
    print(f'Detected headers: {headers}')

    def find_col(keywords):
        """Return the index of the first header that contains any of the keywords."""
        for i, h in enumerate(headers):
            if any(kw in h for kw in keywords):
                return i
        return None

    rank_idx  = find_col(['rank'])
    title_idx = find_col(['title', 'film'])
    gross_idx = find_col(['worldwide', 'gross', 'revenue'])
    year_idx  = find_col(['year'])
    print(f'Column map → rank:{rank_idx}, title:{title_idx}, gross:{gross_idx}, year:{year_idx}')

    if any(idx is None for idx in [rank_idx, title_idx, gross_idx, year_idx]):
        raise ValueError('Could not identify all required columns from the header row.')

    # ── Step 2: Parse data rows, respecting rowspan ───────────────────────────
    # rowspan_map[col_position] = (cell, remaining_rows_to_span)
    rowspan_map = {}
    films = []

    for row in rows[1:]:
        raw_cells = row.find_all(['td', 'th'])
        if not raw_cells:
            continue  # Skip empty rows (e.g. section separators)

        # Rebuild the full column list by filling in rowspan placeholders
        full_row = []
        cell_iter = iter(raw_cells)
        col = 0
        max_col = max(len(headers) + 2, max(rowspan_map.keys(), default=0) + 2)

        while col < max_col:
            if col in rowspan_map:
                # This slot is covered by a cell spanning from a previous row
                span_cell, remaining = rowspan_map[col]
                full_row.append(span_cell)
                if remaining - 1 > 0:
                    rowspan_map[col] = (span_cell, remaining - 1)
                else:
                    del rowspan_map[col]
                col += 1
            else:
                try:
                    cell = next(cell_iter)
                except StopIteration:
                    break  # No more real cells in this row
                rs = int(cell.get('rowspan', 1))
                cs = int(cell.get('colspan', 1))
                if rs > 1:
                    rowspan_map[col] = (cell, rs - 1)
                full_row.append(cell)
                col += cs  # colspan cells occupy multiple positions

        required = max(rank_idx, title_idx, gross_idx, year_idx)
        if len(full_row) <= required:
            continue  # Row too short — skip

        def cell_text(idx):
            return full_row[idx].get_text(strip=True)

        rank_text  = cell_text(rank_idx)
        gross_text = cell_text(gross_idx)
        year_text  = cell_text(year_idx)

        title_cell = full_row[title_idx]
        title = title_cell.get_text(strip=True)
        anchor = title_cell.find('a')
        wiki_link = ('https://en.wikipedia.org' + anchor['href']) if anchor else None

        if title:
            films.append({
                'rank': rank_text,
                'title': title,
                'gross_raw': gross_text,
                'year_raw': year_text,
                'wiki_link': wiki_link,
            })

    return films


raw_films = parse_main_table(soup)
print(f'\nExtracted {len(raw_films)} films from the main table.')
print('Sample entry:', raw_films[0])

Detected headers: ['rank', 'peak', 'title', 'worldwide gross', 'year', 'ref']
Column map → rank:0, title:2, gross:3, year:4

Extracted 50 films from the main table.
Sample entry: {'rank': '1', 'title': 'Avatar', 'gross_raw': '$2,923,710,708', 'year_raw': '2009', 'wiki_link': 'https://en.wikipedia.org/wiki/Avatar_(2009_film)'}


## 4. Enrich Each Film — Fetch Director & Country from Infobox

For each film we visit its individual Wikipedia page and parse the infobox  
to retrieve the **director(s)** and **country of origin**.  
A 0.5-second delay is added between requests to be respectful to the server.

In [6]:
def get_infobox_field(infobox, *field_keywords):
    """Return the text of the first infobox row whose header contains any of the keywords."""
    for row in infobox.find_all('tr'):
        header = row.find('th')
        data = row.find('td')
        if header and data:
            header_text = header.get_text(strip=True).lower()
            if any(kw in header_text for kw in field_keywords):
                # Remove footnote references like [1]
                for sup in data.find_all('sup'):
                    sup.decompose()
                return data.get_text(separator=', ', strip=True)
    return 'N/A'


def fetch_film_details(wiki_url):
    """Visit a film's Wikipedia page and return director and country from the infobox."""
    if not wiki_url:
        return {'director': 'N/A', 'country': 'N/A'}
    try:
        time.sleep(0.5)  # Polite crawl delay
        resp = requests.get(wiki_url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        page = BeautifulSoup(resp.content, 'lxml')

        infobox = page.find('table', {'class': 'infobox'})
        if infobox is None:
            return {'director': 'N/A', 'country': 'N/A'}

        director = get_infobox_field(infobox, 'directed', 'director')
        country = get_infobox_field(infobox, 'country', 'countries')
        return {'director': director, 'country': country}

    except Exception as exc:
        print(f'  Warning: could not fetch details for {wiki_url}: {exc}')
        return {'director': 'N/A', 'country': 'N/A'}


print('Fetching director and country for each film (this may take a minute)...')
for i, film in enumerate(raw_films):
    details = fetch_film_details(film['wiki_link'])
    film.update(details)
    print(f'  [{i+1}/{len(raw_films)}] {film["title"]} — director: {details["director"]}, country: {details["country"]}')

print('\nEnrichment complete.')

Fetching director and country for each film (this may take a minute)...
  [1/50] Avatar — director: James Cameron, country: United States, United Kingdom
  [2/50] Avengers: Endgame — director: Anthony Russo, Joe Russo, country: United States
  [3/50] Avatar: The Way of Water — director: James Cameron, country: United States
  [4/50] Titanic — director: James Cameron, country: United States
  [5/50] Ne Zha 2 — director: Jiaozi, country: China
  [6/50] Star Wars: The Force Awakens — director: J. J. Abrams, country: United States
  [7/50] Avengers: Infinity War — director: Anthony Russo, Joe Russo, country: United States
  [8/50] Spider-Man: No Way Home — director: Jon Watts, country: United States
  [9/50] Zootopia 2† — director: Jared Bush, Byron Howard, country: United States
  [10/50] Inside Out 2 — director: Kelsey Mann, country: United States
  [11/50] Jurassic World — director: Colin Trevorrow, country: United States
  [12/50] The Lion King — director: Jon Favreau, country: United 

## 5. Clean and Normalise the Data

- **Revenue**: strip Wikipedia footnotes (`[1]`), keep the formatted string as-is for display,  
  and also produce a plain numeric version (in billions USD) for sorting.
- **Year**: extract a 4-digit year as an integer.
- **Director / Country**: collapse whitespace and remove any residual footnote markers.

In [7]:
# Matches Wikipedia footnote refs: [1], [a], [# 1], [note 2], etc.
FOOTNOTE_RE = re.compile(r'\[[\w\s#]+\]')


def clean_text(text):
    """Remove footnote markers and normalise whitespace."""
    text = FOOTNOTE_RE.sub('', text)
    return ' '.join(text.split()).strip()


def parse_revenue_numeric(gross_str):
    """
    Parse the USD amount from a revenue string such as:
      '$2,923,710,708'  →  2923710708.0
      'T$2,257,906,828' →  2257906828.0   (T = Wikipedia footnote prefix)
      'NZ$2,215,690,000'→  2215690000.0  (treated same way)
      'F8$1,238,764,765'→  1238764765.0  (F8 = Fast 8 abbreviation prefix)

    Strategy: locate the '$' sign and parse the digits+commas that follow it.
    Fallback: strip all non-numeric characters if no '$' is found.
    """
    cleaned = FOOTNOTE_RE.sub('', gross_str).strip()
    # Primary: extract the number that directly follows a '$' sign
    match = re.search(r'\$([0-9,]+)', cleaned)
    if match:
        try:
            return float(match.group(1).replace(',', ''))
        except ValueError:
            pass
    # Fallback: keep only digits and dots (works for plain numeric strings)
    digits = re.sub(r'[^\d.]', '', cleaned)
    try:
        return float(digits)
    except ValueError:
        return None


def parse_year(year_str):
    """
    Extract a realistic release year (1800–2099) from a string.
    Using a bounded pattern avoids false matches from revenue strings.
    """
    match = re.search(r'\b(1[89]\d{2}|20\d{2})\b', year_str)
    return int(match.group(1)) if match else None


cleaned_films = []
for film in raw_films:
    cleaned_films.append({
        'rank': clean_text(film['rank']),
        'title': clean_text(film['title']),
        'release_year': parse_year(film['year_raw']),
        'director': clean_text(film.get('director', 'N/A')),
        'box_office': clean_text(film['gross_raw']),      # Formatted string kept for display
        'box_office_usd': parse_revenue_numeric(film['gross_raw']),  # Numeric for sorting
        'country': clean_text(film.get('country', 'N/A')),
    })

print(f'Cleaned {len(cleaned_films)} entries.')
print('Sample:', cleaned_films[0])

Cleaned 50 entries.
Sample: {'rank': '1', 'title': 'Avatar', 'release_year': 2009, 'director': 'James Cameron', 'box_office': '$2,923,710,708', 'box_office_usd': 2923710708.0, 'country': 'United States, United Kingdom'}


## 6. Create the SQLite Database and Insert Data

In [8]:
DB_PATH = 'films.db'

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Drop and recreate the table so the notebook is idempotent
cursor.execute('DROP TABLE IF EXISTS films')
cursor.execute('''
    CREATE TABLE films (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        rank         TEXT,
        title        TEXT NOT NULL,
        release_year INTEGER,
        director     TEXT,
        box_office   TEXT,
        box_office_usd REAL,
        country      TEXT
    )
''')
conn.commit()
print(f'Database created: {DB_PATH}')

INSERT_SQL = '''
    INSERT INTO films (rank, title, release_year, director, box_office, box_office_usd, country)
    VALUES (:rank, :title, :release_year, :director, :box_office, :box_office_usd, :country)
'''

cursor.executemany(INSERT_SQL, cleaned_films)
conn.commit()

print(f'Inserted {cursor.rowcount} rows into the films table.')

Database created: films.db
Inserted 50 rows into the films table.


## 7. Verify the Database Contents

In [9]:
cursor.execute('SELECT COUNT(*) FROM films')
total = cursor.fetchone()[0]
print(f'Total films stored: {total}')

print('\nTop 10 by box office revenue:')
cursor.execute('''
    SELECT rank, title, release_year, director, box_office, country
    FROM films
    ORDER BY box_office_usd DESC
    LIMIT 10
''')
for row in cursor.fetchall():
    print(f'  #{row[0]:>3}  {row[1]:<45} ({row[2]})  {row[4]}')

Total films stored: 50

Top 10 by box office revenue:
  #  1  Avatar                                        (2009)  $2,923,710,708
  #  2  Avengers: Endgame                             (2019)  $2,797,501,328
  #  3  Avatar: The Way of Water                      (2022)  $2,334,484,620
  #  4  Titanic                                       (1997)  T$2,257,906,828
  #  5  Ne Zha 2                                      (2025)  NZ$2,215,690,000
  #  6  Star Wars: The Force Awakens                  (2015)  $2,068,223,624
  #  7  Avengers: Infinity War                        (2018)  $2,048,359,754
  #  8  Spider-Man: No Way Home                       (2021)  SM$1,922,598,800
  #  9  Zootopia 2†                                   (2025)  $1,866,647,950
  # 10  Inside Out 2                                  (2024)  $1,698,863,816


## 8. Export to JSON for the Static Web Page

GitHub Pages serves only static files, so the web page loads data from a JSON file  
rather than querying the database directly.

In [10]:
JSON_PATH = 'films.json'

cursor.execute('''
    SELECT id, rank, title, release_year, director, box_office, box_office_usd, country
    FROM films
    ORDER BY box_office_usd DESC
''')
rows = cursor.fetchall()

films_export = [
    {
        'id': r[0],
        'rank': r[1],
        'title': r[2],
        'release_year': r[3],
        'director': r[4],
        'box_office': r[5],
        'box_office_usd': r[6],
        'country': r[7],
    }
    for r in rows
]

with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(films_export, f, ensure_ascii=False, indent=2)

print(f'Exported {len(films_export)} films to {JSON_PATH}.')
conn.close()
print('Database connection closed.')

Exported 50 films to films.json.
Database connection closed.
